# Notebook 01 — Flash / Glare Detection

Detects bright glare regions in an image using OpenCV and marks each one
with a rounded red bounding box.

## Algorithm summary
1. Convert image to grayscale.
2. **Bright mask** — threshold pixels above 225.
3. **Smooth mask** — detect low-texture (flat) regions via Laplacian variance.
4. Combine masks with bitwise-AND → candidate glare regions.
5. Morphologically close + dilate to reconnect glare streaks.
6. Filter contours by area, edge-density, and contrast.
7. Draw a rounded red rectangle + label on each accepted region.

## Usage
Place your image inside `../data/input/` and set `INPUT_IMAGE` below.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os
import time

# ── CONFIG ────────────────────────────────────────────────────────────────────
INPUT_IMAGE  = "../data/input/sample_001.jpg"   # ← change to your image
OUTPUT_DIR   = "../data/output/glares"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
def draw_rounded_rectangle(img, top_left, bottom_right, color, thickness, radius=15):
    """Draw a rectangle with rounded corners using cv2 lines + ellipses."""
    x1, y1 = top_left
    x2, y2 = bottom_right

    # Straight edges
    cv2.line(img, (x1 + radius, y1),        (x2 - radius, y1),        color, thickness)
    cv2.line(img, (x1 + radius, y2),        (x2 - radius, y2),        color, thickness)
    cv2.line(img, (x1,          y1 + radius),(x1,          y2 - radius),color, thickness)
    cv2.line(img, (x2,          y1 + radius),(x2,          y2 - radius),color, thickness)

    # Corners
    cv2.ellipse(img, (x1 + radius, y1 + radius), (radius, radius), 180,  0, 90, color, thickness)
    cv2.ellipse(img, (x2 - radius, y1 + radius), (radius, radius), 270,  0, 90, color, thickness)
    cv2.ellipse(img, (x1 + radius, y2 - radius), (radius, radius),  90,  0, 90, color, thickness)
    cv2.ellipse(img, (x2 - radius, y2 - radius), (radius, radius),   0,  0, 90, color, thickness)

In [ ]:
def detect_and_draw_glare(image_path):
    """
    Detect glare / flash spots in *image_path* and draw annotated bounding boxes.

    Returns
    -------
    output   : annotated BGR image
    glare    : binary glare mask
    original : unmodified BGR image
    """
    img      = cv2.imread(image_path)
    original = img.copy()
    gray     = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # ── Step 1: bright pixels ─────────────────────────────────────────────────
    _, bright = cv2.threshold(gray, 225, 255, cv2.THRESH_BINARY)

    # ── Step 2: smooth (low-texture) pixels ───────────────────────────────────
    lap     = cv2.Laplacian(gray, cv2.CV_64F)
    texture = cv2.convertScaleAbs(lap)
    _, smooth = cv2.threshold(texture, 15, 255, cv2.THRESH_BINARY_INV)

    # ── Step 3: combine ───────────────────────────────────────────────────────
    glare = cv2.bitwise_and(bright, smooth)

    # ── Step 4: morphological cleanup ─────────────────────────────────────────
    glare = cv2.morphologyEx(glare, cv2.MORPH_CLOSE,  np.ones((7, 7), np.uint8))
    glare = cv2.dilate(glare, np.ones((9, 9), np.uint8), iterations=1)

    # ── Step 5: contour filtering + annotation ────────────────────────────────
    contours, _ = cv2.findContours(glare, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    output      = original.copy()
    edges       = cv2.Canny(gray, 50, 150)
    min_area    = 1500

    for cnt in contours:
        area = cv2.contourArea(cnt)
        if area < min_area:
            continue

        x, y, w, h  = cv2.boundingRect(cnt)
        roi          = gray[y:y+h, x:x+w]
        edges_roi    = edges[y:y+h, x:x+w]
        edge_density = np.sum(edges_roi > 0) / (w * h)
        contrast     = roi.std()

        # Skip regions that are textured or high-contrast (not true glare)
        if edge_density > 0.02 or contrast > 25:
            continue

        pad = 12
        x   = max(0, x - pad)
        y   = max(0, y - pad)
        w  += 2 * pad
        h  += 2 * pad

        draw_rounded_rectangle(output, (x, y), (x + w, y + h),
                                color=(0, 0, 255), thickness=3, radius=18)

        label    = "Glare Area"
        font     = cv2.FONT_HERSHEY_SIMPLEX
        tsize    = cv2.getTextSize(label, font, 1, 2)[0]
        text_x   = x + (w - tsize[0]) // 2
        text_y   = y + h + 40
        cv2.putText(output, label, (text_x, text_y), font, 1, (0, 0, 255), 2, cv2.LINE_AA)

    return output, glare, original

In [ ]:
# ── Run ───────────────────────────────────────────────────────────────────────
_t = time.perf_counter()
result, mask, original = detect_and_draw_glare(INPUT_IMAGE)
_elapsed = time.perf_counter() - _t

# ── Save ──────────────────────────────────────────────────────────────────────
filename    = os.path.basename(INPUT_IMAGE)
output_path = os.path.join(OUTPUT_DIR, filename)
cv2.imwrite(output_path, result)
print(f"Saved → {output_path}")
print(f"⏱  Glare detection: {_elapsed:.2f}s")

In [ ]:
# ── Visualise ─────────────────────────────────────────────────────────────────
result_rgb   = cv2.cvtColor(result,   cv2.COLOR_BGR2RGB)
original_rgb = cv2.cvtColor(original, cv2.COLOR_BGR2RGB)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
axes[0].imshow(original_rgb); axes[0].set_title("Original");      axes[0].axis("off")
axes[1].imshow(mask, cmap="gray"); axes[1].set_title("Glare Mask"); axes[1].axis("off")
axes[2].imshow(result_rgb);  axes[2].set_title("Annotated");      axes[2].axis("off")
plt.tight_layout()
plt.show()